In [3]:
import sys
import os

PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

In [4]:
from conllu import parse_incr
from steps.projectivize import is_non_proj
import pandas as pd

In [5]:
def get_non_proj_sentences(gold_path):
    with open(gold_path, "r", encoding="utf-8") as f:
        sent_ids = []
        for tokenlist in parse_incr(f):
            arcs = []
            for token in tokenlist:
                arcs.append((token["head"], token["id"]))
            if is_non_proj(arcs):
                sent_ids.append(tokenlist.metadata["sent_id"])
    return sent_ids

In [7]:
gold_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
non_proj_sents = get_non_proj_sentences(gold_file)

In [8]:
def find_errors(gold_path, pred_path):

    errors = {
        "sent_id":[],
        "token_id": [], 
        "gold_head": [],
        "gold_head_pos":[], 
        "pred_head": [],
        "pred_head_pos":[], 
        "gold_deprel": [],
        "pred_deprel": []
        }    
 
    with open(pred_path, "r", encoding="utf-8") as fpred, \
         open(gold_path, "r", encoding="utf-8") as fgold:
        for sent_pred, sent_gold in zip(parse_incr(fpred), parse_incr(fgold)):
            sent_id = sent_gold.metadata["sent_id"]
            for tok_pred, tok_gold in zip(sent_pred, sent_gold):
                wrong_head = tok_pred["head"] != tok_gold["head"]
                wrong_deprel = tok_pred["deprel"] != tok_gold["deprel"]
                if wrong_head or wrong_deprel:
                    errors["sent_id"].append(sent_id)
                    errors["token_id"].append(tok_pred["id"])
                    errors["gold_head_pos"].append(tok_gold["upos"])
                    errors["pred_head_pos"].append(tok_pred["upos"])

                if wrong_head and not wrong_deprel:
                    errors["gold_head"].append(tok_gold["head"])
                    errors["pred_head"].append(tok_pred["head"])
                    errors["gold_deprel"].append("na")
                    errors["pred_deprel"].append("na")
                elif not wrong_head and wrong_deprel:
                    errors["gold_head"].append("na")
                    errors["pred_head"].append("na")
                    errors["gold_deprel"].append(tok_gold["deprel"])
                    errors["pred_deprel"].append(tok_pred["deprel"])
                elif wrong_head and wrong_deprel:
                    errors["gold_head"].append(tok_gold["head"])
                    errors["pred_head"].append(tok_pred["head"])
                    errors["gold_deprel"].append(tok_gold["deprel"])
                    errors["pred_deprel"].append(tok_pred["deprel"])
    return errors



In [58]:
gold_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
pred_file = os.path.join(PROJECT_ROOT, "predictions", "stanza", "lang=en,bert=finetune,charlm=yes,pretrain=yes,epochs=100,deprojz=yes,matched=yes.conllu")
errors = find_errors(gold_file, pred_file)

In [ ]:
cons_err_df = pd.DataFrame(errors)
cons_err_df

,sent_id,token_id,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
0,english-penn-test-2,23,21,ADJ,19,ADJ,na,na
1,english-penn-test-4,11,5,PUNCT,13,PUNCT,na,na
2,english-penn-test-4,26,5,PUNCT,13,PUNCT,na,na
3,english-penn-test-5,7,na,ADP,na,ADP,advmod,compound:prt
4,english-penn-test-5,10,7,NOUN,6,NOUN,na,na
...,...,...,...,...,...,...,...,...
2441,english-penn-test-2414,10,na,NUM,na,NUM,compound,nummod
2442,english-penn-test-2414,16,3,PUNCT,8,PUNCT,na,na
2443,english-penn-test-2414,17,20,VERB,8,VERB,case,xcomp
2444,english-penn-test-2414,20,3,NOUN,17,NOUN,nmod,dobj


In [60]:
wrong_head_df = err_df.query("gold_head != pred_head")
wrong_head_count = wrong_head_df["token_id"].count()
wrong_head_count

np.int64(1957)

In [37]:
wrong_heads_and_deprels = err_df.query("gold_head != pred_head and gold_deprel != pred_deprel")["token_id"].count()
wrong_heads_and_deprels

np.int64(586)

In [38]:
wrong_deprel_df = err_df.query("gold_deprel != pred_deprel")
wrong_deprel_count = wrong_deprel_df["token_id"].count()
wrong_deprel_count

np.int64(1075)

In [46]:
wrong_head_count - wrong_heads_and_deprels

np.int64(1371)

In [49]:
wrong_deprel_count - wrong_heads_and_deprels

np.int64(489)

In [1]:
import stanza
stanza.download("pl")

c:\Users\hrkwl\.conda\envs\projz\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-29 14:33:09 INFO: Downloaded file to C:\Users\hrkwl\stanza_resources\resources.json
2025-12-29 14:33:09 INFO: Downloading default packages for language: pl (Polish) ...
2025-12-29 14:33:57 INFO: Downloaded file to C:\Users\hrkwl\stanza_resources\pl\default.zip
2025-12-29 14:34:00 INFO: Finished downloading models and saved to C:\Users\hrkwl\stanza_resources
